In [1]:
# importing relevant libraries to complete all tasks
import numpy as np
import pandas as pd
from sklearn.preprocessing import OrdinalEncoder
import matplotlib
matplotlib.use("Agg")
import matplotlib.pyplot as plt
from sklearn.model_selection import train_test_split, learning_curve, StratifiedKFold
from sklearn.tree import DecisionTreeClassifier
from xgboost import XGBClassifier
from pathlib import Path
import os
import json

In [2]:
# Get the current notebook's directory
notebook_dir = Path().resolve() 

In [3]:
# csv file name declaration
data_file_1 = 'financial_fraud_detection_dataset.csv'
data_file_2 = 'FraudShield_Banking_Data (1).csv'

In [4]:
# Build the relative path to the CSV file
csv_file_path_1 = os.path.join(notebook_dir, data_file_1)
csv_file_path_2 = os.path.join(notebook_dir, data_file_2)

In [5]:
finanical_data_raw = pd.read_csv(csv_file_path_1, sep=',', decimal='.', header=0)

In [6]:
banking_data_raw = pd.read_csv(csv_file_path_2, sep=',', decimal='.', header=0)

In [7]:
# Before the data preparation, whole set of data will be cloned for further processing
finanical_data_process = finanical_data_raw.copy()

In [8]:
# Before the data preparation, whole set of data will be cloned for further processing
banking_data_process = banking_data_raw.copy()

In [9]:
def _add_time_features(df, dt_col):
    df["txn_day_of_week"] = df[dt_col].dt.dayofweek.astype("Int64")
    df["is_weekend"] = df["txn_day_of_week"].isin([5, 6]).astype("Int64")
    df["Hour"] = df[dt_col].dt.hour.astype("Int64")
    df["is_night"] = df["Hour"].between(0, 5).astype("Int64")
    return df

In [10]:
finanical_data_process["timestamp"] = pd.to_datetime(finanical_data_process["timestamp"], format="ISO8601")

In [11]:
finanical_data_process["time_since_last_txn_missing"] = finanical_data_process["time_since_last_transaction"].isna().astype("Int64")

In [12]:
finanical_data_process = _add_time_features(finanical_data_process, "timestamp")

In [13]:
finanical_data_process["log_amount"] = np.log1p(finanical_data_process["amount"])

In [14]:
fd_y = finanical_data_process["is_fraud"].astype(bool)

In [15]:
# keep human-readable value for fairness analysis
location_raw = finanical_data_process["location"].copy()

In [16]:
feature_cols = [
    "amount", "transaction_type", "merchant_category", "location", "device_used",
    "time_since_last_transaction", "spending_deviation_score", "velocity_score",
    "geo_anomaly_score", "payment_channel", "time_since_last_txn_missing",
    "txn_day_of_week", "is_weekend", "Hour", "is_night", "log_amount",
]

In [17]:
fd_X = finanical_data_process[feature_cols].copy()

In [18]:
FD_CATEGORICAL = ["transaction_type", "merchant_category", "location", "device_used", "payment_channel"]

In [19]:
enc = OrdinalEncoder()
fd_X[FD_CATEGORICAL] = enc.fit_transform(fd_X[FD_CATEGORICAL])

In [20]:
# Combine Date and Time strings with a separating space
combined_series = banking_data_process['Transaction_Date'] + ' ' + banking_data_process['Transaction_Time']

# Convert to datetime format specifying the explicit layout
banking_data_process['Transaction_DateTime'] = pd.to_datetime(
    combined_series, 
    format='%Y-%m-%d %H:%M', 
    errors='coerce'
)

In [21]:
BD_CATEGORICAL = [
    "Transaction_Type", "Merchant_Category", "Transaction_Location",
    "Customer_Home_Location", "Card_Type", "Is_International_Transaction",
    "Is_New_Merchant", "Unusual_Time_Transaction",
]

In [22]:
for col in BD_CATEGORICAL:
    banking_data_process[col] = banking_data_process[col].fillna("Unknown")

In [23]:
banking_data_process = banking_data_process.dropna(subset=["Fraud_Label", "Transaction_DateTime"])

In [24]:
banking_data_process = _add_time_features(banking_data_process, "Transaction_DateTime")

In [25]:
home_location_raw = banking_data_process["Customer_Home_Location"].copy()

In [26]:
bd_y = banking_data_process["Fraud_Label"].map({"Fraud": True, "Normal": False}).astype(bool)

In [27]:
feature_cols = [
    "Transaction_Amount (in Million)", "Transaction_Type", "Merchant_Category",
    "Transaction_Location", "Customer_Home_Location", "Distance_From_Home",
    "Card_Type", "Account_Balance (in Million)", "Daily_Transaction_Count",
    "Weekly_Transaction_Count", "Avg_Transaction_Amount (in Million)",
    "Max_Transaction_Last_24h (in Million)", "Is_International_Transaction",
    "Is_New_Merchant", "Failed_Transaction_Count", "Unusual_Time_Transaction",
    "Previous_Fraud_Count", "txn_day_of_week", "is_weekend", "Hour", "is_night",
]

In [28]:
bd_X = banking_data_process[feature_cols].copy()

In [29]:
enc = OrdinalEncoder()
bd_X[BD_CATEGORICAL] = enc.fit_transform(bd_X[BD_CATEGORICAL])

In [30]:
CV_FOLDS = 5
TRAIN_SIZES = np.linspace(0.1, 1.0, 7)  # 10%, 25%, 40%, 55%, 70%, 85%, 100%
RANDOM_STATE = 42

In [31]:
# Best hyperparameters found by RandomizedSearchCV in Part 1 (reused, not re-tuned)
FD_DT_PARAMS = {"min_samples_split": 50, "min_samples_leaf": 5, "max_depth": 6,
                 "criterion": "gini", "class_weight": "balanced"}
FD_XGB_PARAMS = {"subsample": 1.0, "n_estimators": 100, "min_child_weight": 3,
                  "max_depth": 4, "learning_rate": 0.03, "colsample_bytree": 0.8}
BD_DT_PARAMS = {"min_samples_split": 2, "min_samples_leaf": 5, "max_depth": 4,
                 "criterion": "entropy", "class_weight": "balanced"}
BD_XGB_PARAMS = {"subsample": 0.8, "n_estimators": 300, "min_child_weight": 3,
                  "max_depth": 4, "learning_rate": 0.01, "colsample_bytree": 0.8}

In [32]:
def compute_curve(model, X, y, label):
    cv = StratifiedKFold(n_splits=CV_FOLDS, shuffle=True, random_state=RANDOM_STATE)
    sizes, train_scores, val_scores = learning_curve(
        model, X, y,
        train_sizes=TRAIN_SIZES, cv=cv, scoring="average_precision",
        n_jobs=1, shuffle=True, random_state=RANDOM_STATE,
    )
    print(f"{label}: done. sizes={sizes.tolist()}")
    return {
        "train_sizes": sizes.tolist(),
        "train_mean": train_scores.mean(axis=1).tolist(),
        "train_std": train_scores.std(axis=1).tolist(),
        "val_mean": val_scores.mean(axis=1).tolist(),
        "val_std": val_scores.std(axis=1).tolist(),
    }

In [33]:
def plot_curve(ax, result, title):
    sizes = np.array(result["train_sizes"])
    tr_mean, tr_std = np.array(result["train_mean"]), np.array(result["train_std"])
    va_mean, va_std = np.array(result["val_mean"]), np.array(result["val_std"])
 
    ax.plot(sizes, tr_mean, "o-", color="#1f77b4", label="Training score", markersize=6, zorder=3)
    ax.fill_between(sizes, tr_mean - tr_std, tr_mean + tr_std, alpha=0.15, color="#1f77b4")
    ax.plot(sizes, va_mean, "o-", color="#d62728", label="Cross-validation score", markersize=6, zorder=3)
    ax.fill_between(sizes, va_mean - va_std, va_mean + va_std, alpha=0.15, color="#d62728")
 
    # Annotate each point with its PR-AUC value
    for x, y in zip(sizes, tr_mean):
        ax.annotate(f"{y:.4f}", (x, y), textcoords="offset points", xytext=(0, 8),
                    fontsize=7, color="#1f77b4", ha="center")
    for x, y in zip(sizes, va_mean):
        ax.annotate(f"{y:.4f}", (x, y), textcoords="offset points", xytext=(0, -12),
                    fontsize=7, color="#d62728", ha="center")
 
    ax.set_title(title, fontsize=11)
    ax.set_xlabel("Training set size (rows)")
    ax.set_ylabel("PR-AUC (average precision)")
    ax.legend(loc="best", fontsize=8)
    ax.grid(alpha=0.3)

In [34]:
fd_X_train, _, fd_y_train, _ = train_test_split(
    fd_X, fd_y, test_size=0.2, stratify=fd_y, random_state=RANDOM_STATE
)

In [35]:
fd_scale_pos_weight = (fd_y_train == 0).sum() / (fd_y_train == 1).sum()

In [36]:
results = {}

In [37]:
print("Computing Dataset 1 - Decision Tree learning curve...")
fd_dt = DecisionTreeClassifier(random_state=RANDOM_STATE, **FD_DT_PARAMS)
results["fd_dt"] = compute_curve(fd_dt, fd_X_train, fd_y_train, "Dataset1-DT")

Computing Dataset 1 - Decision Tree learning curve...
Dataset1-DT: done. sizes=[320000, 800000, 1280000, 1759999, 2240000, 2720000, 3200000]


In [38]:
print("Computing Dataset 1 - XGBoost learning curve...")
fd_xgb = XGBClassifier(
    random_state=RANDOM_STATE, scale_pos_weight=fd_scale_pos_weight,
    eval_metric="aucpr", n_jobs=1, **FD_XGB_PARAMS,
)
results["fd_xgb"] = compute_curve(fd_xgb, fd_X_train, fd_y_train, "Dataset1-XGB")

Computing Dataset 1 - XGBoost learning curve...
Dataset1-XGB: done. sizes=[320000, 800000, 1280000, 1759999, 2240000, 2720000, 3200000]


In [39]:
bd_X_train, _, bd_y_train, _ = train_test_split(
    bd_X, bd_y, test_size=0.2, stratify=bd_y, random_state=RANDOM_STATE
)

In [40]:
bd_scale_pos_weight = (bd_y_train == 0).sum() / (bd_y_train == 1).sum()

In [41]:
print("Computing Dataset 2 - Decision Tree learning curve...")
bd_dt = DecisionTreeClassifier(random_state=RANDOM_STATE, **BD_DT_PARAMS)
results["bd_dt"] = compute_curve(bd_dt, bd_X_train, bd_y_train, "Dataset2-DT")

Computing Dataset 2 - Decision Tree learning curve...
Dataset2-DT: done. sizes=[3198, 7997, 12795, 17593, 22392, 27190, 31989]


In [42]:
print("Computing Dataset 2 - XGBoost learning curve...")
bd_xgb = XGBClassifier(
    random_state=RANDOM_STATE, scale_pos_weight=bd_scale_pos_weight,
    eval_metric="aucpr", n_jobs=1, **BD_XGB_PARAMS,
)
results["bd_xgb"] = compute_curve(bd_xgb, bd_X_train, bd_y_train, "Dataset2-XGB")

Computing Dataset 2 - XGBoost learning curve...
Dataset2-XGB: done. sizes=[3198, 7997, 12795, 17593, 22392, 27190, 31989]


In [43]:
fig, axes = plt.subplots(2, 2, figsize=(12, 9))
plot_curve(axes[0, 0], results["fd_dt"], "Financial Transactions Dataset - Decision Tree\n(full training set, 4,000,000 rows)")
plot_curve(axes[0, 1], results["fd_xgb"], "Financial Transactions Dataset - XGBoost\n(full training set, 4,000,000 rows)")
plot_curve(axes[1, 0], results["bd_dt"], "Banking Fraud Detection Dataset - Decision Tree\n(full training set, 39,987 rows)")
plot_curve(axes[1, 1], results["bd_xgb"], "Banking Fraud Detection Dataset - XGBoost\n(full training set, 39,987 rows)")
fig.suptitle("Learning Curves: PR-AUC vs. Training Set Size", fontsize=14)
fig.tight_layout(rect=[0, 0, 1, 0.96])

In [44]:
fig.savefig("learning_curves.png", dpi=150)